# Trace Diagnosis — Experiment Notebook

This Notebook configures and runs experiments on Who&When, TRAIL, and MAADBench. Five dataset IDs are supported:

- **Who&When:** `who_when_algorithm`, `who_when_handcrafted`
- **TRAIL:** `trail_gaia`, `trail_swe_bench`
- **MAADBench:** `maadbench`

**Recommended workflow:** Imports and paths → common settings → dataset-specific options → validate the experiment plan → convert graphs if needed → inspect graphs → run diagnosis → inspect checkpoints → evaluate → view metrics.

- Graph conversion: Read the original data and generate graphs without calling a model.
- Diagnosis: Read graphs, call the model, and save checkpoints. This stage may incur API charges.
- Evaluation: Evaluate existing checkpoints without calling a model.

`RUN_CONVERSION`, `RUN_DIAGNOSIS`, and `RUN_EVALUATION` are all `False` by default. Complete configuration and validation before enabling each required stage individually.

> Place this Notebook in the project root containing `independent_apps/`. If it is located elsewhere, set `PROJECT_ROOT_OVERRIDE` in Section 1. Select a Jupyter kernel with the required Python/Conda dependencies installed. Keep API keys in an external API configuration JSON file, not in the Notebook.

Before running, install the dependencies with `python -m pip install -r requirements.txt` and prepare the datasets using [README.md](README.md). Copy `api_config.example.json` to a private `api_config.json` and configure the model endpoint.


## 1. Imports and Project Path (Run Once)

This section locates the project directory and prepares the execution environment.

If the project cannot be located automatically, set `PROJECT_ROOT_OVERRIDE` in the code cell to the project root containing `independent_apps/`. Ensure that the Jupyter kernel uses a Python/Conda environment with the project dependencies installed.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib
import json
import re
import sys

try:
    import pandas as pd
except ImportError:
    pd = None

# If the Notebook is outside the project root, specify an absolute path, e.g., Path("path/to/Trace-diagnosis")
PROJECT_ROOT_OVERRIDE = None

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = (Path.cwd(), *Path.cwd().parents)
    PROJECT_ROOT = next(
        (candidate for candidate in candidates if (candidate / "independent_apps" / "shared" / "cli.py").is_file()),
        None,
    )
if PROJECT_ROOT is None or not (PROJECT_ROOT / "independent_apps" / "shared" / "cli.py").is_file():
    raise FileNotFoundError("Cannot find independent_apps/shared/cli.py; set PROJECT_ROOT_OVERRIDE.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from independent_apps.shared import cli as project_cli
from independent_apps.shared.settings import RuntimeSettings

print("Project:", PROJECT_ROOT)
print("Python:", sys.executable)
print("CLI:", project_cli.__file__)
print("Available dataset IDs:", ", ".join(project_cli.DATASET_IDS))

## 2. Common Experiment Settings (Main Configuration)

Configure the dataset, model, API connection, input/output paths, execution scope, and common diagnosis options here.

### 2.1 Data and Execution Scope

- `DATASET`: Choose `who_when_algorithm`, `who_when_handcrafted`, `trail_gaia`, `trail_swe_bench`, or `maadbench`.
- `INPUT_OVERRIDE`: Override the default dataset input path; `None` uses the project default.
- `OUTPUT_ROOT`: Root directory for Notebook outputs.
- `END`: Number of records to diagnose. `2` diagnoses the first two records; `None` diagnoses every record in the selected scope.
- `GRAPH_BUILD_LIMIT`: Limit the number of records converted to graphs, independently of `END`. MAADBench builds a full graph index, and `MAAD_SCOPE` determines the diagnosis scope.
- `GRAPH_OVERRIDE`, `CHECKPOINT_OVERRIDE`, `EVALUATION_OVERRIDE`: Specify paths to reuse existing files; `None` uses automatically generated paths.
- `EXPERIMENT_TAG`: Experiment label. Use different labels for different models or key experiment conditions.

### 2.2 Model and Execution Stages

- `MODEL_ID`: Diagnosis model ID; `None` uses the model in the API configuration or the project default.
- `API_CONFIG`: Path to an external API configuration JSON file.
- `PROXY`: HTTP proxy; `None` uses the default setting, and an empty string clears the explicit proxy.
- `RUN_CONVERSION`: Enable graph conversion without model calls.
- `RUN_DIAGNOSIS`: Enable diagnosis, which makes model calls and may incur API charges.
- `RUN_EVALUATION`: Evaluate existing predictions without model calls.
- `REBUILD_GRAPH`: Rebuild an existing graph. Do not rebuild at the same path if a corresponding checkpoint already exists.

### 2.3 Cascade Call Mode

- `CASCADE_ENABLED=True`: Enable cascade inspection. `False` disables cascade inspection and uses local-only diagnosis.
- `CASCADE_CALL_MODE="combined"`: Use a joint model call for local inspection and cascade attribution at a node, returning both results.
- `CASCADE_CALL_MODE="separate"`: Perform local inspection and cascade attribution in separate model calls at a node.
- These call patterns refer to the local and cascade portions of node inspection, not the total number of requests for a trace. Other stages, such as final-hypothesis initialization, may require additional calls.

`SEPARATE_ORDER` applies to separate mode for all three benchmarks:

- `local_then_cascade`: Perform local inspection first and pass the local result to cascade attribution.
- `cascade_then_local`: Perform cascade attribution first and pass its result to local inspection as evidence.

Who&When defaults to `local_then_cascade` to preserve the existing experiment settings. TRAIL and MAADBench default to `cascade_then_local`. Either order can be selected explicitly.

### 2.4 Traversal and Prompt Context

- `TRAVERSAL_MODE`: `auto`, `bfs`, or `dfs`. `auto` currently resolves to DFS for Who&When and BFS for TRAIL and MAADBench.
- `PROMPT_CONTEXT_POLICY`:
  - `legacy`: Use the existing upstream-context selection method.
  - `ranked_budget`: Rank candidate upstream context and select content within the character budget.
  - `prioritized_two_kinds`: TRAIL only; select up to two non-empty edge-type groups in the priority order `data_flow`, `execution_flow`, `parent`, `other`.
- `MAX_CAUSES`: Maximum number of candidate causes retained and propagated.
- `MAX_PROMPT_UPSTREAM_CHARS`: Character budget for upstream context; applies to prompt strategies that use budget-based selection.

For Who&When, `MAX_CAUSES`, `MAX_PROMPT_UPSTREAM_CHARS`, `PROMPT_CONTEXT_POLICY` and `WHO_WHEN_STAGE2_CANDIDATES` apply to Combined only. Separate retains its own internal settings.

### 2.5 Label Visibility

- `NO_GROUND_TRUTH=False`: Do not explicitly remove the top-level graph `ground_truth` field. Set to `True` to exclude that field from model-visible data; recommended for blind diagnosis. The task reference answer and the diagnostic target labels are distinct; this switch is not a guarantee that every reference embedded in raw trace text has been removed.
- `INCLUDE_EVAL_LABELS=False`: Do not save evaluation labels in prediction records. Set to `True` to retain those labels for auditing; top-level evaluation labels remain excluded from diagnosis.

### 2.6 Graph Edge Types (`EDGE_TYPES`)

`None` uses the full default edge set of the selected dataset's converter. All three benchmarks support a non-empty subset of their allowed edge types, for example `EDGE_TYPES = ("execution_flow", "data_flow")`.

| Dataset | Allowed edge types | Default edge set |
|---|---|---|
| Who&When | `sequential_flow`, `execution_flow`, `data_flow` | All three types |
| TRAIL | `contains`, `execution_flow`, `data_flow` | All three types |
| MAADBench | `sequential_flow`, `execution_flow`, `data_flow` | All three types |

Edge definitions:

- `sequential_flow`: Temporal order between adjacent steps or events.
- `execution_flow`: Execution or control dependencies, such as reasoning to action or a parent action to a tool call.
- `data_flow`: The output of an earlier node is used as input by a later node.
- `contains`: Containment between parent and child spans.

Changing the edge set generates a different graph path. Regenerate the graph and use its corresponding checkpoint.


In [ ]:
# Dataset: who_when_algorithm, who_when_handcrafted, trail_gaia, trail_swe_bench, maadbench
DATASET = "who_when_algorithm"

# Model and connection settings. Do not put plaintext API keys in the Notebook.
MODEL_ID = "gpt-5-mini"            # None uses the external API configuration
API_CONFIG = PROJECT_ROOT / "api_config.json"  # Private copy of api_config.example.json; use the correct provider endpoint
PROXY = None                    # None uses the CLI/environment default; "" clears the explicit proxy

# Paths. Use dataset registry defaults unless explicitly overridden.
INPUT_OVERRIDE = None             # e.g., Path("path/to/Data/gpt-4.1")
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "notebook_independent"
CHECKPOINT_OVERRIDE = None        # Absolute path to an existing checkpoint for evaluation or resumption
GRAPH_OVERRIDE = None             # Absolute path to an existing graph
EVALUATION_OVERRIDE = None        # Absolute path to an existing evaluation report
EXPERIMENT_TAG = "example_run"  # Use distinct labels for different diagnosis/converter experiments

# Execution controls: disabled by default to avoid accidental API charges from Run All.
RUN_CONVERSION = False
RUN_DIAGNOSIS = False
RUN_EVALUATION = False
REBUILD_GRAPH = False              # True allows reconversion; rebuilding is blocked if a checkpoint exists

END = 2                              # Diagnose two records first; set to None after verification
GRAPH_BUILD_LIMIT = None           # None converts all graphs; MAADBench always builds the full index

CASCADE_ENABLED = True
CASCADE_CALL_MODE = "separate"     # Select separate or combined local/cascade inspection
SEPARATE_ORDER = "local_then_cascade" if DATASET.startswith("who_when") else "cascade_then_local"  # Either order may be selected explicitly
TRAVERSAL_MODE = "auto"            # auto / bfs / dfs; auto is BFS for TRAIL and MAADBench
PROMPT_CONTEXT_POLICY = "legacy"
MAX_CAUSES = 8                         # Who&When: Combined only; other benchmarks use their own cause-state policies
MAX_PROMPT_UPSTREAM_CHARS = 3000
NO_GROUND_TRUTH = False            # True is recommended for blind diagnosis; removes the top-level ground_truth field
INCLUDE_EVAL_LABELS = False        # Save evaluation labels in output records for auditing

# None uses the full default converter edge set.
# TRAIL accepts any non-empty subset of ("contains", "execution_flow", "data_flow")
# MAADBench accepts any non-empty subset of ("sequential_flow", "execution_flow", "data_flow")
# Who&When accepts any non-empty subset of ("sequential_flow", "execution_flow", "data_flow")
EDGE_TYPES = None

## 3. Dataset-Specific Options

Configure only the parameters corresponding to the selected `DATASET`. Options set to `None` generally use the project defaults.

### 3.1 Who&When

- `WHO_WHEN_STAGE2_CANDIDATES`: Maximum number of root-cause candidate nodes considered during Combined Stage 2 selection; default 20. The Separate implementation retains its own internal candidate settings.
- `WHO_WHEN_INCLUDE_FULL_ANALYSIS`: Save complete analysis details in prediction records; disabled by default.

### 3.2 TRAIL

**Cause state: `TRAIL_CAUSE_STATE`**

- `H0`: Conventional cause state. Causes are merged primarily by cause ID, so the same cause arriving from different branches may share a state.
- `H1`: Path-aware cause state. Maintain separate states by `(Cause ID, Incoming Path)`, allowing the same cause to be tracked independently along different paths without branches overwriting each other's states. H1 uses `execution_flow` and `data_flow` dependency edges for cause propagation.

**Scheduling scope: `TRAIL_SCHEDULE_SCOPE`**

- `legacy`: Retain the existing scheduling scope associated with final nodes.
- `all_nodes`: Include every graph node in the diagnosis schedule.
- `None`: Use the default `legacy`.

**Final hypothesis policy: `TRAIL_FINAL_HYPOTHESIS_POLICY`**

- `gated`: Initialize upstream candidate causes only when the final output is judged to have failed.
- `retain_candidates`: Retain trace-supported upstream candidate causes even if the final output is not judged to have failed.
- `None`: Use the default `gated`.

**Request and token budgets**

- `TRAIL_MAX_COMPLETION_TOKENS`: Maximum completion tokens per model response; `None` uses the project default.
- `TRAIL_API_REQUEST_LIMIT`: Maximum number of API requests; `None` means no explicit client-side request-count cap.
- `TRAIL_REQUEST_TOKEN_BUDGET`: Enable request token-budget management.
- `TRAIL_REQUEST_CONTEXT_LIMIT`: Request context token limit.
- `TRAIL_REQUEST_HARD_PROJECTED_LIMIT`: Hard limit on projected tokens before sending a request.
- `TRAIL_REQUEST_COMPLETION_RESERVE`: Completion tokens reserved for model output.
- `TRAIL_REQUEST_SAFETY_FACTOR`: Safety multiplier for token estimates.
- `TRAIL_REQUEST_TOKENIZER`: Tokenizer for token estimation; `None` uses the project default.
- `TRAIL_ENABLE_SCORING`: Enable optional TRAIL scoring output.
- `TRAIL_USE_LLM_ERROR_SYNTHESIS`: Enable the additional LLM-based error synthesis step.

### 3.3 MAADBench

**Diagnosis scope: `MAAD_SCOPE`**

- `all`: Diagnose all traces in the graph index; evaluation still matches test actions against the official test manifest.
- `official_test`: Diagnose only traces in the official test split.
- `official_dev`: Diagnose the development split constructed from the official training portion by grouping rooms, and evaluate with the corresponding development manifest.

**Cause state: `MAAD_CAUSE_STATE`**

- `legacy`: Conventional cause-state management, merging and updating states by cause ID.
- `path`: Maintain separate cause states by `(Cause ID, Incoming Path)`, retaining evidence and states along different paths. Path mode requires cascade inspection.

`MAAD_MAX_CAUSE_PATH_STATES` limits the number of cause-path states in path mode (default 4096). Some paths may be truncated if the limit is reached; inspect path coverage.

**Evaluation profile: `MAAD_EVALUATION_PROFILE`**

- `legacy`: Include only complete prediction records with `SUCCESS` / `OK` status.
- `local_complete_cause_partial`: Additionally include path-mode records with complete local predictions but `PARTIAL` cause-path coverage. Every action must have a valid local prediction; not all partial records qualify. For example, if all 10 actions have local predictions but some cause paths remain incomplete, the strict profile excludes the entire trace, while the permissive profile can include those 10 actions. If even one action lacks a local prediction, both profiles exclude that partial trace. Retain `legacy` for primary results and report the permissive profile and coverage separately.

**Evaluation score source: `MAAD_EVALUATION_SCORE_SOURCE`**

- `final`: Read each action's `anomaly_score`. This works with both cause-state modes and is the recommended default.
- `local`: Read `local_prediction.anomaly_score` or `local_anomaly_score`. In the current path mode, `final` and `local` store the same local score, while legacy mode generally does not store a separate local field. This option is intended primarily for historical score comparisons; it does not rerun local-only diagnosis.

**Threshold policy: `MAAD_THRESHOLD_POLICY`**

- `fixed`: Use the official fixed-threshold rule. `MAAD_ANOMALY_FRACTION` controls the anomaly fraction (default 0.1).
- `oracle`: Use test-set ground-truth labels to determine the top-k threshold. This is an oracle comparison, not an unlabeled threshold available at deployment time.

**Data filtering and development settings**

- `MAAD_DOMAIN`: `gsm-hard` or `livecodebench`; select the official evaluation domain.
- `MAAD_TRACE_TYPE`: `tool` or `no_tool`; select the tool-use condition.
- `MAAD_SOURCE_MODELS`: Filter by the model that generated the original traces. An empty list uses the model corresponding to the input directory.
- `MAAD_SOURCE_TEMPERATURES`: Filter by original trace-generation temperature. An empty list defaults to 0.0.
- `MAAD_OFFICIAL_ROOT`: Official MAADBench repository root containing `benchmark_AD/utils.py`. Discovery starts from the input path by default.
- `MAAD_DEV_SIZE`: Fraction of the official training portion used for the development split.
- `MAAD_DEV_SEED`: Development split random seed.
- `MAAD_DEV_TRACE_LIMIT`: Maximum number of traces for a development pilot; `None` uses the full development split.
- `MAAD_DEV_PILOT_SEED`: Random seed for selecting a smaller pilot subset from the development pool.
- `MAAD_DUMP_VISIBLE_CONTEXT`: Export model-visible context for debugging; disabled by default.


In [ ]:
# ---------- Who&When ----------
WHO_WHEN_STAGE2_CANDIDATES = 20
WHO_WHEN_INCLUDE_FULL_ANALYSIS = False

# ---------- TRAIL ----------
TRAIL_CAUSE_STATE = "H0"          # H0 / H1
TRAIL_SCHEDULE_SCOPE = None       # None / legacy / all_nodes
TRAIL_FINAL_HYPOTHESIS_POLICY = None  # None / gated / retain_candidates
TRAIL_MAX_COMPLETION_TOKENS = 26000    # None uses the project default of 32768
TRAIL_API_REQUEST_LIMIT = None        # None means no explicit client-side request-count cap; set a positive integer to impose a cap
TRAIL_REQUEST_TOKEN_BUDGET = True
TRAIL_REQUEST_CONTEXT_LIMIT = 272000
TRAIL_REQUEST_HARD_PROJECTED_LIMIT = 272000
TRAIL_REQUEST_COMPLETION_RESERVE = 26000
TRAIL_REQUEST_SAFETY_FACTOR = 1.10
TRAIL_REQUEST_TOKENIZER = None
TRAIL_ENABLE_SCORING = False
TRAIL_USE_LLM_ERROR_SYNTHESIS = False

# ---------- MAADBench ----------
MAAD_SCOPE = "all"               # all / official_test / official_dev
MAAD_CAUSE_STATE = "legacy"      # legacy / path
MAAD_MAX_CAUSE_PATH_STATES = 4096
MAAD_EVALUATION_PROFILE = "legacy"  # legacy / local_complete_cause_partial
MAAD_EVALUATION_SCORE_SOURCE = "final"  # final / local
MAAD_THRESHOLD_POLICY = "fixed"  # fixed / oracle; use a consistent policy when comparing metrics
MAAD_ANOMALY_FRACTION = 0.1
MAAD_DOMAIN = "gsm-hard"         # gsm-hard / livecodebench (explicitly set to match the existing evaluation default)
MAAD_TRACE_TYPE = "tool"         # tool / no_tool
MAAD_SOURCE_MODELS = []          # e.g., ["gpt-4.1"]
MAAD_SOURCE_TEMPERATURES = []    # e.g., [0.0]
MAAD_OFFICIAL_ROOT = None        # Official repository root containing benchmark_AD/utils.py
MAAD_DEV_SIZE = None             # None uses the project default
MAAD_DEV_SEED = 106              # This field is passed to int(); set a positive integer to avoid int(None)
MAAD_DEV_TRACE_LIMIT = None
MAAD_DEV_PILOT_SEED = 107
MAAD_DUMP_VISIBLE_CONTEXT = False

## 4. Construct Paths and Run Parameters (No Editing Required)

This section automatically generates graph, checkpoint, and evaluation paths and an experiment ID from the dataset, model, input path, and experiment parameters.

- Graph paths depend on the dataset, edge set, and graph conversion scope.
- Checkpoint and evaluation paths are separated by experiment configuration to prevent different models or diagnosis settings from overwriting one another.
- After changing the settings in Sections 2 or 3, rerun this section to refresh the paths and experiment ID.
- To use existing graph, checkpoint, or evaluation files, set the corresponding `*_OVERRIDE` paths in Section 2.


In [ ]:
if DATASET not in project_cli.DATASET_ROUTES:
    raise ValueError(f"Unknown DATASET={DATASET!r}; allowed values: {project_cli.DATASET_IDS}")

family = ("who_when" if DATASET.startswith("who_when") else "trail" if DATASET.startswith("trail") else "maadbench")
reg = importlib.import_module(f"independent_apps.{family}.registry")
spec = reg.get_dataset(DATASET)
INPUT_PATH = Path(INPUT_OVERRIDE if INPUT_OVERRIDE is not None else spec.input_path).expanduser()

allowed_edges = {
    "who_when": ("sequential_flow", "execution_flow", "data_flow"),
    "trail": ("contains", "execution_flow", "data_flow"),
    "maadbench": ("sequential_flow", "execution_flow", "data_flow"),
}
if SEPARATE_ORDER not in {"local_then_cascade", "cascade_then_local"}:
    raise ValueError("SEPARATE_ORDER must be local_then_cascade or cascade_then_local")
if CASCADE_CALL_MODE not in {"separate", "combined"}:
    raise ValueError("CASCADE_CALL_MODE must be separate or combined")
if family in allowed_edges and EDGE_TYPES is not None:
    if not EDGE_TYPES or len(set(EDGE_TYPES)) != len(EDGE_TYPES) or not set(EDGE_TYPES) <= set(allowed_edges[family]):
        raise ValueError(f"Invalid {family} EDGE_TYPES: {EDGE_TYPES}; allowed: {allowed_edges[family]}")
if family == "maadbench" and MAAD_CAUSE_STATE == "path" and not CASCADE_ENABLED:
    raise ValueError("MAADBench path mode requires CASCADE_ENABLED=True")
if END is not None and END < 1:
    raise ValueError("END must be None or a positive integer")
if GRAPH_BUILD_LIMIT is not None and GRAPH_BUILD_LIMIT < 1:
    raise ValueError("GRAPH_BUILD_LIMIT must be None or a positive integer")

# Graph filenames depend on the dataset, edge set, and conversion scope, not model settings.
edge_label = "all_edges" if EDGE_TYPES is None else "+".join(sorted(EDGE_TYPES))
build_label = "full" if family == "maadbench" or GRAPH_BUILD_LIMIT is None else f"first{GRAPH_BUILD_LIMIT}"
# Use the current converter; retain the __current suffix to reuse existing graph files.
graph_converter_label = "__current" if family == "who_when" else ""
GRAPH_PATH = Path(GRAPH_OVERRIDE) if GRAPH_OVERRIDE is not None else (
    OUTPUT_ROOT / "graphs" / DATASET
    / f"{edge_label}__{build_label}{graph_converter_label}.jsonl"
)

identity = {
    "dataset": DATASET,
    "model": MODEL_ID,
    "api_config_path": str(Path(API_CONFIG).resolve()),
    "proxy_setting": PROXY,
    "input": str(INPUT_PATH.resolve()),
    "graph": str(GRAPH_PATH.resolve()),
    "experiment_tag": EXPERIMENT_TAG,
    "cascade": CASCADE_ENABLED,
    "call_mode": CASCADE_CALL_MODE,
    "separate_order": (SEPARATE_ORDER if CASCADE_CALL_MODE == "separate" and (family != "who_when" or (CASCADE_ENABLED and SEPARATE_ORDER == "cascade_then_local")) else None),  # Preserve old Who&When Local-first checkpoint identity.
    "traversal": TRAVERSAL_MODE,
    "context": PROMPT_CONTEXT_POLICY,
    "max_causes": MAX_CAUSES,
    "max_prompt_upstream_chars": MAX_PROMPT_UPSTREAM_CHARS,
    "no_ground_truth": NO_GROUND_TRUTH,
    "include_eval_labels": INCLUDE_EVAL_LABELS,
    "edges": list(EDGE_TYPES) if EDGE_TYPES else None,
    "who_when_diagnosis_mode": CASCADE_CALL_MODE if family == "who_when" else None,
    "who_when": [WHO_WHEN_STAGE2_CANDIDATES, WHO_WHEN_INCLUDE_FULL_ANALYSIS] if family == "who_when" else None,
    "trail": [TRAIL_CAUSE_STATE, TRAIL_SCHEDULE_SCOPE, TRAIL_FINAL_HYPOTHESIS_POLICY,
              TRAIL_MAX_COMPLETION_TOKENS, TRAIL_API_REQUEST_LIMIT, TRAIL_REQUEST_TOKEN_BUDGET,
              TRAIL_REQUEST_CONTEXT_LIMIT, TRAIL_REQUEST_HARD_PROJECTED_LIMIT,
              TRAIL_REQUEST_COMPLETION_RESERVE, TRAIL_REQUEST_SAFETY_FACTOR,
              TRAIL_REQUEST_TOKENIZER, TRAIL_ENABLE_SCORING, TRAIL_USE_LLM_ERROR_SYNTHESIS] if family == "trail" else None,
    "maadbench": [MAAD_SCOPE, MAAD_CAUSE_STATE, MAAD_MAX_CAUSE_PATH_STATES,
                  MAAD_EVALUATION_PROFILE, MAAD_EVALUATION_SCORE_SOURCE, MAAD_THRESHOLD_POLICY,
                  MAAD_ANOMALY_FRACTION, MAAD_DOMAIN, MAAD_TRACE_TYPE, MAAD_SOURCE_MODELS,
                  MAAD_SOURCE_TEMPERATURES, MAAD_OFFICIAL_ROOT, MAAD_DEV_SIZE,
                  MAAD_DEV_SEED, MAAD_DEV_TRACE_LIMIT, MAAD_DEV_PILOT_SEED,
                  MAAD_DUMP_VISIBLE_CONTEXT] if family == "maadbench" else None,
}
fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:12]
safe_model = re.sub(r"[^a-zA-Z0-9._-]", "_", MODEL_ID or "model_from_config")
safe_tag = re.sub(r"[^a-zA-Z0-9._-]", "_", EXPERIMENT_TAG)
strategy_label = (
    CASCADE_CALL_MODE if family == "who_when"
    else CASCADE_CALL_MODE if CASCADE_ENABLED else "local_only"
)
run_id = f"{safe_tag}_{safe_model}_{strategy_label}_{TRAVERSAL_MODE}_{fingerprint}"
run_dir = OUTPUT_ROOT / "runs" / DATASET / run_id
CHECKPOINT_FILE = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE is not None else run_dir / "predictions.json"
EVALUATION_OUTPUT = Path(EVALUATION_OVERRIDE) if EVALUATION_OVERRIDE is not None else run_dir / "evaluation.json"

print("Dataset:", DATASET, "| owner:", family)
print("Model:", MODEL_ID or "from api config")
print("Input:", INPUT_PATH, "[exists]" if INPUT_PATH.exists() else "[missing]")
print("Graph:", GRAPH_PATH, "[exists]" if GRAPH_PATH.exists() else "[not created]")
print("Checkpoint:", CHECKPOINT_FILE, "[exists]" if CHECKPOINT_FILE.exists() else "[not created]")
print("Evaluation:", EVALUATION_OUTPUT)
print("Experiment ID:", run_id)
print("Diagnosis END:", END, "| Graph build:", build_label)
if family == "who_when":
    print("Who&When diagnosis mode:", CASCADE_CALL_MODE)
    if CASCADE_ENABLED and CASCADE_CALL_MODE == "separate":
        print("Separate order:", SEPARATE_ORDER)

## 5. Validate the Experiment Plan (No Model Calls)

This section validates the parameter combination for the selected dataset and builds the arguments for graph conversion, diagnosis, and evaluation.

It only parses and validates arguments; it does not convert graphs, call models, or calculate metrics. If validation fails, correct the settings in Sections 2 or 3 before proceeding.


In [ ]:
def _pair(flag, value):
    return [] if value is None else [flag, str(value)]

def _flag_set(stage):
    argv = ["--dataset", DATASET, "--input", str(INPUT_PATH),
            "--graph-path", str(GRAPH_PATH), "--checkpoint-file", str(CHECKPOINT_FILE)]
    if MODEL_ID is not None:
        argv += ["--llm-model", MODEL_ID]
    if stage == "diagnosis":
        argv += ["--api-config", str(API_CONFIG)]
    if PROXY is not None:
        argv += ["--proxy", str(PROXY)]
    argv += ["--max-causes", str(MAX_CAUSES), "--traversal-mode", TRAVERSAL_MODE]
    argv += _pair("--max-prompt-upstream-chars", MAX_PROMPT_UPSTREAM_CHARS)
    # Select the diagnosis mode for the current benchmark.
    # Preserve other projects' previous behavior when cascade is disabled.
    if family == "who_when" or CASCADE_ENABLED:
        argv += ["--cascade-call-mode", CASCADE_CALL_MODE]
    if not CASCADE_ENABLED:
        argv += ["--no-cascade"]
    if NO_GROUND_TRUTH:
        argv += ["--no-ground-truth"]
    if INCLUDE_EVAL_LABELS:
        argv += ["--include-eval-labels"]

    # Project-owned options. No other project's flags appear in this run.
    argv += ["--prompt-context-policy", PROMPT_CONTEXT_POLICY]
    if family == "who_when":
        argv += ["--max-stage2-candidates", str(WHO_WHEN_STAGE2_CANDIDATES)]
        if WHO_WHEN_INCLUDE_FULL_ANALYSIS:
            argv += ["--include-full-analysis"]
    if CASCADE_ENABLED and CASCADE_CALL_MODE == "separate":
        argv += ["--separate-order", SEPARATE_ORDER]
    if EDGE_TYPES is not None and (family == "who_when" or stage != "evaluate"):
        argv += ["--edge-types", *EDGE_TYPES]
    if family == "trail":
        argv += ["--trail-cause-state-mode", TRAIL_CAUSE_STATE]
        argv += _pair("--trail-schedule-scope", TRAIL_SCHEDULE_SCOPE)
        argv += _pair("--trail-final-hypothesis-policy", TRAIL_FINAL_HYPOTHESIS_POLICY)
        argv += _pair("--trail-max-completion-tokens", TRAIL_MAX_COMPLETION_TOKENS)
        argv += _pair("--trail-api-request-limit", TRAIL_API_REQUEST_LIMIT)
        if TRAIL_REQUEST_TOKEN_BUDGET:
            argv += ["--trail-request-token-budget"]
        argv += _pair("--trail-request-context-limit", TRAIL_REQUEST_CONTEXT_LIMIT)
        argv += _pair("--trail-request-hard-projected-limit", TRAIL_REQUEST_HARD_PROJECTED_LIMIT)
        argv += _pair("--trail-request-completion-reserve", TRAIL_REQUEST_COMPLETION_RESERVE)
        argv += _pair("--trail-request-safety-factor", TRAIL_REQUEST_SAFETY_FACTOR)
        argv += _pair("--trail-request-tokenizer", TRAIL_REQUEST_TOKENIZER)
        if TRAIL_ENABLE_SCORING:
            argv += ["--enable-scoring"]
        if TRAIL_USE_LLM_ERROR_SYNTHESIS:
            argv += ["--use-llm-error-synthesis"]
    if family == "maadbench":
        argv += ["--maadbench-diagnosis-scope", MAAD_SCOPE]
        argv += ["--maadbench-cause-state-mode", MAAD_CAUSE_STATE]
        argv += ["--maadbench-max-cause-path-states", str(MAAD_MAX_CAUSE_PATH_STATES)]
        argv += ["--maadbench-evaluation-profile", MAAD_EVALUATION_PROFILE]
        argv += ["--maadbench-evaluation-score-source", MAAD_EVALUATION_SCORE_SOURCE]
        argv += ["--maadbench-threshold-policy", MAAD_THRESHOLD_POLICY]
        argv += ["--maadbench-anomaly-fraction", str(MAAD_ANOMALY_FRACTION)]
        argv += _pair("--maadbench-domain", MAAD_DOMAIN)
        argv += _pair("--maadbench-trace-type", MAAD_TRACE_TYPE)
        argv += _pair("--maadbench-official-root", MAAD_OFFICIAL_ROOT)
        argv += _pair("--maadbench-dev-size", MAAD_DEV_SIZE)
        argv += _pair("--maadbench-dev-seed", MAAD_DEV_SEED)
        argv += _pair("--maadbench-dev-trace-limit", MAAD_DEV_TRACE_LIMIT)
        argv += _pair("--maadbench-dev-pilot-seed", MAAD_DEV_PILOT_SEED)
        if MAAD_DUMP_VISIBLE_CONTEXT:
            argv += ["--maadbench-dump-visible-context"]
        for model in MAAD_SOURCE_MODELS:
            argv += ["--maadbench-source-model", str(model)]
        for temperature in MAAD_SOURCE_TEMPERATURES:
            argv += ["--maadbench-source-temperature", str(temperature)]
    if stage == "convert":
        argv += ["--convert-only"]
        if family != "maadbench" and GRAPH_BUILD_LIMIT is not None:
            argv += ["--end", str(GRAPH_BUILD_LIMIT)]
    elif stage == "diagnosis":
        argv += ["--skip-convert"]
        if END is not None:
            argv += ["--end", str(END)]
    elif stage == "evaluate":
        argv += ["--evaluate-only", "--evaluation-output", str(EVALUATION_OUTPUT)]
    else:
        raise ValueError(stage)
    return argv

CONVERT_ARGS = _flag_set("convert")
DIAGNOSE_ARGS = _flag_set("diagnosis")
EVALUATE_ARGS = _flag_set("evaluate")

# Validate with the project parser without executing the pipeline.
parser = project_cli.build_parser_for_dataset(DATASET)
for label, argv in (("Conversion", CONVERT_ARGS), ("Diagnosis", DIAGNOSE_ARGS), ("Evaluation", EVALUATE_ARGS)):
    parsed = parser.parse_args(argv)
    print(f"[VALID] {label}: {parsed.dataset} | flags={len(argv)}")

print("\nSelected graph edge set:", EDGE_TYPES or "converter default (all edges)")
print("Convert enabled:", RUN_CONVERSION, "| Diagnose enabled:", RUN_DIAGNOSIS, "| Evaluate enabled:", RUN_EVALUATION)
print("Only Diagnosis may call an LLM API; current switch:", RUN_DIAGNOSIS)

## 6. Graph Conversion (Offline; No Model Calls)

Set `RUN_CONVERSION=True` and execute this cell to read the source data and generate graphs.

- `EDGE_TYPES=None` uses the full default edge set for the selected dataset.
- All three benchmarks allow custom subsets of the permitted edge types in Section 2.
- `GRAPH_BUILD_LIMIT` controls graph conversion scope, whereas `END` controls subsequent diagnosis scope.
- MAADBench builds a full graph index; `MAAD_SCOPE` determines which traces are diagnosed.

Existing graphs are reused by default rather than overwritten. Set `REBUILD_GRAPH=True` explicitly to request conversion again. If a corresponding checkpoint already exists, the Notebook prevents rebuilding at the same graph path to avoid mismatching old predictions with new graphs.


In [ ]:
if RUN_CONVERSION:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Source data not found: {INPUT_PATH}")
    if GRAPH_PATH.exists() and not REBUILD_GRAPH:
        print("[REUSE GRAPH]", GRAPH_PATH)
    else:
        if GRAPH_PATH.exists() and CHECKPOINT_FILE.exists():
            raise RuntimeError("A checkpoint already exists. Rebuilding the corresponding graph at the same path is blocked. Use a different EXPERIMENT_TAG/path and back up existing results.")
        GRAPH_PATH.parent.mkdir(parents=True, exist_ok=True)
        print("[CONVERT START]", DATASET)
        project_cli.main(CONVERT_ARGS)
        print("[CONVERT END]", GRAPH_PATH, "exists=", GRAPH_PATH.is_file())
else:
    print("RUN_CONVERSION=False: no conversion or API requests were made.")

## 7. Inspect Graphs (Read-Only)

This section reads existing graph files without calling a model or modifying any files.

Inspect the graph record count, trace IDs, node and edge counts, and actual edge types. Before running diagnosis, ensure that the graph matches the selected dataset, input data, and edge configuration.


In [ ]:
if GRAPH_PATH.is_file():
    rows = []
    graph_count = 0
    with GRAPH_PATH.open("r", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if not line.strip():
                continue
            graph_count += 1
            graph = json.loads(line)
            if len(rows) < 8:
                rows.append({
                    "index": i,
                    "trace_id": str(graph.get("trace_id", ""))[:64],
                    "nodes": len(graph.get("nodes") or []),
                    "edges": len(graph.get("edges") or []),
                    "edge_kinds": ", ".join(sorted({str(e.get("kind")) for e in graph.get("edges", []) if isinstance(e, dict)})),
                })
    print("Graph records:", graph_count)
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        for row in rows:
            print(row)
else:
    print("Graph not found. Run Section 6 or set GRAPH_OVERRIDE:", GRAPH_PATH)

## 8. Run Diagnosis (**Makes Model API Calls**)

Set `RUN_DIAGNOSIS=True` and execute this cell. Before proceeding, confirm that:

- The graph file exists and matches the current input data and edge set.
- `API_CONFIG` points to a readable external JSON configuration file.
- The model ID, proxy, and request budgets are correct.
- `END` matches the intended small-scale test or full diagnosis scope.

`CASCADE_CALL_MODE=combined` uses a joint call to return local and cascade results; `separate` uses separate calls. The order in separate mode is controlled by `SEPARATE_ORDER` in Section 2.

Existing checkpoints support resumption under matching configurations. Use a new checkpoint path when changing the model, graph edge set, cause state, diagnosis mode, separate order, or MAADBench split.

A single trace may require multiple model requests. Check prompt sizes and API costs even when `END=2`.


In [ ]:
if RUN_DIAGNOSIS:
    if not GRAPH_PATH.is_file():
        raise FileNotFoundError(f"Diagnosis requires a graph file. Run conversion first: {GRAPH_PATH}")
    if not Path(API_CONFIG).is_file():
        raise FileNotFoundError(f"API configuration file not found (do not put keys in the Notebook): {API_CONFIG}")
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    print("[DIAGNOSIS] dataset:", DATASET, "model:", MODEL_ID or "config", "END:", END)
    print("[CHECKPOINT]", CHECKPOINT_FILE)
    if CHECKPOINT_FILE.exists():
        print("[RESUME] The dataset runner determines which records are complete and which to retry")
    result = project_cli.main(DIAGNOSE_ARGS)
    print("[DIAGNOSIS RETURNED]", "records=" + str(len(result)) if isinstance(result, list) else type(result).__name__)
else:
    print("RUN_DIAGNOSIS=False: no model API requests were made.")

## 9. Inspect Checkpoints and Failed Records (Read-Only)

This section reads existing checkpoints and summarizes execution statuses and failures without making model requests or modifying checkpoints.

- `SUCCESS`: The record completed.
- `PARTIAL`: Some results are available, but diagnosis or coverage is incomplete.
- `FAILED` / `ERROR`: The record did not complete successfully.

For MAADBench path mode, inspect both local prediction status and cause-path coverage. Complete local predictions do not necessarily imply complete cause-path attribution. Execution status is not a measure of diagnostic accuracy.


In [ ]:
if CHECKPOINT_FILE.is_file():
    raw = json.loads(CHECKPOINT_FILE.read_text(encoding="utf-8-sig"))
    records = raw if isinstance(raw, list) else [raw] if isinstance(raw, dict) else []
    state_counts = Counter(str(item.get("status", "UNKNOWN")) for item in records if isinstance(item, dict))
    local_counts = Counter(str(item.get("local_prediction_status", "UNKNOWN")) for item in records if isinstance(item, dict))
    print("Saved records:", len(records), "| status:", dict(state_counts), "| local_prediction_status:", dict(local_counts))
    samples = [
        {"index": i, "trace_id": str(rec.get("trace_id", ""))[:64],
         "status": rec.get("status", "UNKNOWN"), "local_prediction_status": rec.get("local_prediction_status", "UNKNOWN"),
         "error": str(rec.get("error", ""))[:110]}
        for i, rec in enumerate(records[:20]) if isinstance(rec, dict)
    ]
    if pd is not None:
        display(pd.DataFrame(samples))
    else:
        for item in samples:
            print(item)
else:
    print("No checkpoint available:", CHECKPOINT_FILE)

## 10. Evaluate Existing Predictions (No Model Calls)

Set `RUN_EVALUATION=True` and execute this cell. It reads the existing checkpoint, graph, and original reference data to calculate evaluation metrics without rerunning diagnosis or calling an LLM.

Official MAADBench test/development evaluation requires access to the official benchmark utilities. If automatic discovery fails, set `MAAD_OFFICIAL_ROOT` in Section 3.

If the checkpoint contains only partial predictions from a run such as `END=2`, the metrics are suitable for workflow checks but must not be presented as full-dataset paper results.

For Who&When, use the original labeled JSON directory as the input/reference path. A single JSON/JSONL file is supported for conversion only.


In [ ]:
if RUN_EVALUATION:
    if not CHECKPOINT_FILE.is_file():
        raise FileNotFoundError(f"Prediction file not found: {CHECKPOINT_FILE}")
    if not GRAPH_PATH.is_file():
        raise FileNotFoundError(f"Corresponding graph file not found: {GRAPH_PATH}")
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Original reference input not found: {INPUT_PATH}")
    EVALUATION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    print("[EVALUATION]", DATASET, "=>", EVALUATION_OUTPUT)
    project_cli.main(EVALUATE_ARGS)
else:
    print("RUN_EVALUATION=False: no evaluation was performed.")

## 11. View Evaluation Metrics and Compare Runs on the Same Dataset (Read-Only)

This section reads the saved `evaluation.json`, displays metrics, and scans previous evaluation reports for the selected dataset. It does not make model calls or rerun evaluation.

- Who&When: Agent Accuracy, Step Accuracy.
- TRAIL: Category F1, Location Accuracy, Joint.
- MAADBench: Action-level F1, Accuracy, AUC, Balanced Accuracy, and additional metrics such as direct error localization.

Compare numerical results only under matching dataset, evaluation split, threshold policy, score source, and coverage conditions, where applicable.


In [ ]:
def numeric_metrics(payload, prefix=""):
    """Flatten numeric metrics in JSON without expanding full predictions or data."""
    found = {}
    if isinstance(payload, dict):
        for key, value in payload.items():
            name = f"{prefix}.{key}" if prefix else key
            if isinstance(value, dict):
                found.update(numeric_metrics(value, name))
            elif isinstance(value, (int, float)) and not isinstance(value, bool):
                found[name] = value
    return found

if EVALUATION_OUTPUT.is_file():
    evaluation = json.loads(EVALUATION_OUTPUT.read_text(encoding="utf-8-sig"))
    print("Evaluation report:", EVALUATION_OUTPUT)
    metrics = numeric_metrics(evaluation)
    if pd is not None:
        display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
    else:
        print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print("No evaluation results available:", EVALUATION_OUTPUT)

# Scan previous Notebook evaluation files for the same dataset without running new experiments.
root = OUTPUT_ROOT / "runs" / DATASET
summary = []
if root.exists():
    for path in sorted(root.glob("*/evaluation.json")):
        try:
            payload = json.loads(path.read_text(encoding="utf-8-sig"))
            row = {"run_id": path.parent.name}
            row.update(numeric_metrics(payload))
            summary.append(row)
        except (OSError, json.JSONDecodeError):
            pass
if summary:
    print("\nSaved evaluation reports for", DATASET, ":", len(summary))
    if pd is not None:
        display(pd.DataFrame(summary))
    else:
        for item in summary:
            print(item)
else:
    print("No saved results are available for comparison.")

## 12. Usage Notes

1. Start with `END=2` to verify the workflow. For full experiments, use `END=None` and confirm evaluation coverage.
2. Use a new checkpoint path when changing the model, graph edge set, cause state, diagnosis mode, separate order, or MAADBench split.
3. For blind diagnosis, set `NO_GROUND_TRUTH=True` and keep `INCLUDE_EVAL_LABELS=False`.
4. `MAAD_SCOPE=all` diagnoses all graph records; select `official_test` to diagnose only the official test split.
5. `MAAD_EVALUATION_PROFILE=local_complete_cause_partial` changes which partial records qualify for evaluation. Report the profile used.
6. `MAAD_THRESHOLD_POLICY=oracle` uses evaluation ground-truth labels to determine the threshold and is suitable only as an oracle comparison. Explicitly report the threshold policy used for primary results.
7. After changing `EDGE_TYPES` or `SEPARATE_ORDER`, rerun Sections 4–5. Do not manually reuse checkpoints from a different configuration.

For manually overridden checkpoints, verify the model, graph and experiment settings before resuming. The default paths protect most common configurations, but do not guarantee compatibility of arbitrary existing files.
